# **Código Principal**

In [2]:
from abc import ABC, abstractmethod
from typing import Optional


# ════════════════════════════════════════════════════════════════════
#  INTERFACES
# ════════════════════════════════════════════════════════════════════

class Exibivel(ABC):
    @abstractmethod
    def exibir(self) -> None: ...

class Avaliavel(ABC):
    @abstractmethod
    def avaliar(self, avaliacao: "Avaliacao") -> None: ...

class Pesquisavel(ABC):
    @abstractmethod
    def buscar_por_genero(self, genero: str) -> list: ...


# ════════════════════════════════════════════════════════════════════
#  PLATAFORMA
# ════════════════════════════════════════════════════════════════════

class Plataforma(Exibivel):
    _total_plataformas: int = 0

    def __init__(self, nome: str, fabricante: str) -> None:
        self._nome: str = nome
        self._fabricante: str = fabricante
        self._online: bool = True
        Plataforma._total_plataformas += 1

    @property
    def nome(self) -> str:
        return self._nome

    @nome.setter
    def nome(self, valor: str) -> None:
        if not valor.strip():
            raise ValueError("Nome não pode ser vazio.")
        self._nome = valor

    @property
    def fabricante(self) -> str:
        return self._fabricante

    @fabricante.setter
    def fabricante(self, valor: str) -> None:
        self._fabricante = valor

    @property
    def online(self) -> bool:
        return self._online

    @online.setter
    def online(self, valor: bool) -> None:
        self._online = valor

    @classmethod
    def total(cls) -> int:
        return cls._total_plataformas

    def exibir(self) -> None:
        status = "🟢 Online" if self._online else "🔴 Offline"
        print(f"🖥️  {self._nome} ({self._fabricante}) — {status}")

    def __repr__(self) -> str:
        return f"Plataforma({self._nome!r}, {self._fabricante!r})"


# ════════════════════════════════════════════════════════════════════
#  AVALIACAO
# ════════════════════════════════════════════════════════════════════

class Avaliacao(Exibivel):
    NOTA_MIN: int = 0
    NOTA_MAX: int = 10

    def __init__(self, nota: float, comentario: str) -> None:
        self._nota: float = nota
        self._comentario: str = comentario
        self._recomenda: bool = nota >= 7.0

    @property
    def nota(self) -> float:
        return self._nota

    @nota.setter
    def nota(self, valor: float) -> None:
        if not (self.NOTA_MIN <= valor <= self.NOTA_MAX):
            raise ValueError(f"Nota deve estar entre {self.NOTA_MIN} e {self.NOTA_MAX}.")
        self._nota = valor
        self._recomenda = valor >= 7.0

    @property
    def comentario(self) -> str:
        return self._comentario

    @comentario.setter
    def comentario(self, valor: str) -> None:
        self._comentario = valor

    @property
    def recomenda(self) -> bool:
        return self._recomenda

    def exibir(self) -> None:
        rec = "👍 Recomendado" if self._recomenda else "👎 Não recomendado"
        print(f"   ⭐ Nota: {self._nota:.1f}/10 — {self._comentario} ({rec})")

    def __repr__(self) -> str:
        return f"Avaliacao(nota={self._nota}, recomenda={self._recomenda})"


# ════════════════════════════════════════════════════════════════════
#  JOGO  (classe base abstrata)
# ════════════════════════════════════════════════════════════════════

class Jogo(Exibivel, Avaliavel, ABC):
    _total_jogos: int = 0

    def __init__(self, titulo: str, genero: str, plataforma: Plataforma,
                 horas_jogadas: float = 0.0) -> None:
        self._titulo: str = titulo
        self._genero: str = genero
        self._plataforma: Plataforma = plataforma
        self._horas_jogadas: float = horas_jogadas
        self._zerado: bool = False
        self._avaliacao: Optional[Avaliacao] = None
        self._favorito: bool = False
        Jogo._total_jogos += 1

    @property
    def titulo(self) -> str:
        return self._titulo

    @titulo.setter
    def titulo(self, valor: str) -> None:
        if not valor.strip():
            raise ValueError("Título não pode ser vazio.")
        self._titulo = valor

    @property
    def genero(self) -> str:
        return self._genero

    @genero.setter
    def genero(self, valor: str) -> None:
        self._genero = valor

    @property
    def horas_jogadas(self) -> float:
        return self._horas_jogadas

    @horas_jogadas.setter
    def horas_jogadas(self, valor: float) -> None:
        if valor < 0:
            raise ValueError("Horas não podem ser negativas.")
        self._horas_jogadas = valor

    @property
    def zerado(self) -> bool:
        return self._zerado

    @zerado.setter
    def zerado(self, valor: bool) -> None:
        self._zerado = valor

    @property
    def avaliacao(self) -> Optional[Avaliacao]:
        return self._avaliacao

    @property
    def plataforma(self) -> Plataforma:
        return self._plataforma

    @property
    def favorito(self) -> bool:
        return self._favorito

    @favorito.setter
    def favorito(self, valor: bool) -> None:
        self._favorito = valor

    def avaliar(self, avaliacao: Avaliacao) -> None:
        self._avaliacao = avaliacao

    @abstractmethod
    def tipo(self) -> str: ...

    def exibir(self) -> None:
        status = "✅ Zerado" if self._zerado else "⚠️ Em andamento"
        fav    = " [❤️]" if self._favorito else ""
        print(
            f"● {self._titulo}{fav} | {self._genero} | {self.tipo()} | "
            f"{self._plataforma.nome} | {self._horas_jogadas:.0f}h | {status}"
        )
        if self._avaliacao:
            self._avaliacao.exibir()

    @classmethod
    def total(cls) -> int:
        return cls._total_jogos

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({self._titulo!r})"


# ════════════════════════════════════════════════════════════════════
#  SUBCLASSES
# ════════════════════════════════════════════════════════════════════

class JogoSingle(Jogo):
    def __init__(self, titulo, genero, plataforma, horas_jogadas=0.0,
                 duracao_campanha: float = 0.0, runs: int = 0) -> None:
        super().__init__(titulo, genero, plataforma, horas_jogadas)
        self._duracao_campanha: float = duracao_campanha
        self._runs: int = runs

    @property
    def duracao_campanha(self) -> float:
        return self._duracao_campanha

    @duracao_campanha.setter
    def duracao_campanha(self, valor: float) -> None:
        self._duracao_campanha = max(0.0, valor)

    @property
    def runs(self) -> int:
        return self._runs

    @runs.setter
    def runs(self, valor: int) -> None:
        self._runs = max(0, valor)

    def progresso(self) -> float:
        if self._duracao_campanha == 0:
            return 0.0
        return min(100.0, (self._horas_jogadas / self._duracao_campanha) * 100)

    def tipo(self) -> str:
        return "Single-player"

    def exibir(self) -> None:
        super().exibir()
        if self._duracao_campanha:
            print(f"   📊 Progresso da campanha: {self.progresso():.1f}%")
        if self._runs:
            print(f"   🔁 Runs totais: {self._runs}")
        print()


class JogoMultiplayer(Jogo):
    def __init__(self, titulo, genero, plataforma, horas_jogadas=0.0,
                 partidas: int = 0) -> None:
        super().__init__(titulo, genero, plataforma, horas_jogadas)
        self._partidas: int = partidas

    @property
    def partidas(self) -> int:
        return self._partidas

    @partidas.setter
    def partidas(self, valor: int) -> None:
        if valor < 0:
            raise ValueError("Número de partidas não pode ser negativo.")
        self._partidas = valor

    def registrar_partida(self, horas: float = 0.5) -> None:
        self._partidas += 1
        self._horas_jogadas += horas

    def tipo(self) -> str:
        return "Multiplayer"

    def exibir(self) -> None:
        super().exibir()
        print(f"   🎯 Partidas jogadas: {self._partidas}")
        print()


# ════════════════════════════════════════════════════════════════════
#  BIBLIOTECA
# ════════════════════════════════════════════════════════════════════

class Biblioteca(Pesquisavel):
    def __init__(self) -> None:
        self._jogos: list[Jogo] = []

    @property
    def jogos(self) -> list[Jogo]:
        return list(self._jogos)

    def adicionar_jogo(self, jogo: Jogo) -> None:
        if any(j.titulo == jogo.titulo for j in self._jogos):
            return
        self._jogos.append(jogo)

    def remover_jogo(self, titulo: str) -> None:
        for j in self._jogos:
            if j.titulo == titulo:
                print(f"🗑️  Removendo '{j.titulo}'...")
                self._jogos.remove(j)
                del j
                return
        print(f"❌ Nenhum jogo removido.")

    def listar_todos(self) -> None:
        if not self._jogos:
            print("Biblioteca vazia.")
            return
        for j in self._jogos:
            j.exibir()

    def listar_zerados(self) -> None:
        resultado = [j for j in self._jogos if j.zerado]
        if not resultado:
            print("Nenhum jogo zerado ainda.")
            return
        for j in resultado:
            j.exibir()

    def listar_em_andamento(self) -> None:
        resultado = [j for j in self._jogos if not j.zerado]
        if not resultado:
            print("Nenhum jogo em andamento.")
            return
        for j in resultado:
            j.exibir()

    def listar_favoritos(self) -> None:
        resultado = [j for j in self._jogos if j.favorito]
        if not resultado:
            print("Nenhum favorito marcado.")
            return
        for j in resultado:
            j.exibir()

    def buscar_por_genero(self, genero: str) -> list[Jogo]:
        """Busca exata por gênero (case-insensitive)."""
        return [j for j in self._jogos if j.genero.lower() == genero.lower()]

    def buscar_por_genero_parcial(self, termo: str) -> list[Jogo]:
        """Busca parcial — encontra 'FPS' dentro de 'FPS Tático', por exemplo."""
        return [j for j in self._jogos if termo.lower() in j.genero.lower()]

    def listar_por_genero(self, genero: str, parcial: bool = False) -> None:
        """Exibe os resultados de uma busca por gênero formatada."""
        resultado = (
            self.buscar_por_genero_parcial(genero)
            if parcial
            else self.buscar_por_genero(genero)
        )
        modo = "(busca parcial)" if parcial else ""
        header = f"═══ BUSCA POR GÊNERO: \"{genero}\" {modo}"
        if resultado:
            print(f"\n{header} — {len(resultado)} resultado(s) ═══")
            for j in resultado:
                j.exibir()
        else:
            print(f"\n{header} — nenhum resultado ═══\n")

    def generos_disponiveis(self) -> list[str]:
        """Retorna lista de gêneros únicos presentes na biblioteca."""
        return sorted({j.genero for j in self._jogos})

    def total_horas(self) -> float:
        return sum(j.horas_jogadas for j in self._jogos)

    def media_horas(self) -> float:
        if not self._jogos:
            return 0.0
        return self.total_horas() / len(self._jogos)

    def jogo_mais_jogado(self) -> Optional[Jogo]:
        if not self._jogos:
            return None
        return max(self._jogos, key=lambda j: j.horas_jogadas)

    def __len__(self) -> int:
        return len(self._jogos)


# ════════════════════════════════════════════════════════════════════
#  USUARIO
# ════════════════════════════════════════════════════════════════════

class Usuario(Exibivel):
    _total_usuarios: int = 0

    def __init__(self, nome: str, username: str, email: str = "") -> None:
        self._nome: str = nome
        self._username: str = username
        self._email: str = email
        self._biblioteca: Biblioteca = Biblioteca()
        self._nivel: int = 1
        Usuario._total_usuarios += 1

    @property
    def nome(self) -> str:
        return self._nome

    @nome.setter
    def nome(self, valor: str) -> None:
        if not valor.strip():
            raise ValueError("Nome não pode ser vazio.")
        self._nome = valor

    @property
    def username(self) -> str:
        return self._username

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, valor: str) -> None:
        self._email = valor

    @property
    def biblioteca(self) -> Biblioteca:
        return self._biblioteca

    @property
    def nivel(self) -> int:
        return self._nivel

    @classmethod
    def total(cls) -> int:
        return cls._total_usuarios

    @staticmethod
    def validar_username(username: str) -> bool:
        return username.isalnum() and len(username) >= 3

    def _calcular_nivel(self) -> int:
        horas = self._biblioteca.total_horas()
        if horas < 10:   return 1
        if horas < 50:   return 2
        if horas < 150:  return 3
        if horas < 300:  return 4
        return 5

    def exibir(self) -> None:
        self._nivel = self._calcular_nivel()
        lib = self._biblioteca
        print(f"\n{'═'*45}")
        print(f"  👤 {self._nome}  (@{self._username})")
        if self._email:
            print(f"  📧 {self._email}")
        print(f"  🏆 Nível: {self._nivel}  |  🎮 Jogos: {len(lib)}")
        print(f"  ⏱️  Total: {lib.total_horas():.0f}h  |  "
              f"Média: {lib.media_horas():.1f}h/jogo")
        top = lib.jogo_mais_jogado()
        if top:
            print(f"  🥇 Mais jogado: {top.titulo} ({top.horas_jogadas:.0f}h)")
        print(f"{'═'*45}\n")

    def __repr__(self) -> str:
        return f"Usuario({self._nome!r}, @{self._username})"


# ════════════════════════════════════════════════════════════════════
#  EXECUÇÃO PRINCIPAL
# ════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Plataformas ──────────────────────────────────────────────────
    ps5      = Plataforma("PS5",             "Sony")
    steam    = Plataforma("Steam",           "Valve")
    xbox     = Plataforma("Xbox",            "Microsoft")
    nintendo = Plataforma("Nintendo Switch", "Nintendo")

    # ── Usuário ──────────────────────────────────────────────────────
    lucas = Usuario("Lucas Mendes", "lucasgamer", "lucas@email.com")

    # ── Jogos ────────────────────────────────────────────────────────
    j1 = JogoSingle("Hollow Knight", "Metroidvania", steam,
                    horas_jogadas=40, duracao_campanha=35)
    j1.zerado   = True
    j1.favorito = True
    j1.avaliar(Avaliacao(9, "Obra de arte, vale cada centavo."))

    j2 = JogoSingle("God of War", "Ação/Aventura", ps5,
                    horas_jogadas=25, duracao_campanha=20)
    j2.zerado = True
    j2.avaliar(Avaliacao(10, "Narrativa impecável."))

    j3 = JogoSingle("Hades", "Roguelike", steam,
                    horas_jogadas=60, runs=47)
    j3.zerado   = True
    j3.favorito = True
    j3.avaliar(Avaliacao(10, "Impossível parar de jogar."))

    j4 = JogoSingle("Resident Evil 4", "Survival Horror", xbox,
                    horas_jogadas=15, duracao_campanha=16)
    j4.zerado = False

    j5 = JogoMultiplayer("Valorant", "FPS Tático", steam,
                         horas_jogadas=120, partidas=300)
    j5.zerado = False

    # ── Adicionar à biblioteca ───────────────────────────────────────
    lib = lucas.biblioteca
    for jogo in (j1, j2, j3, j4, j5):
        lib.adicionar_jogo(jogo)

    lib.adicionar_jogo(j1)  # duplicata

    j5.registrar_partida(horas=0.8)

    # ── Remover da biblioteca ────────────────────────────────────────
    jogo_remover = ''
    lib.remover_jogo(jogo_remover)

    # ── Listagens ────────────────────────────────────────────────────
    print("\n═══ TODOS OS JOGOS ═══")
    lib.listar_todos()

    print("\n═══ EM ANDAMENTO ═══")
    lib.listar_em_andamento()

    print("\n═══ ZERADOS ═══")
    lib.listar_zerados()

    print("\n═══ FAVORITOS ═══")
    lib.listar_favoritos()

    # ── Busca por gênero ─────────────────────────────────────────────
    print("\n═══ GÊNEROS DISPONÍVEIS NA BIBLIOTECA ═══")
    for g in lib.generos_disponiveis():
        print(f"   🏷️  {g}")

    lib.listar_por_genero("Survival Horror")        
    lib.listar_por_genero("FPS Tático")             
    lib.listar_por_genero("Roguelike")             
    lib.listar_por_genero("FPS", parcial=True)      
    lib.listar_por_genero("RPG")                    

    lucas.exibir()

    print(f"📊 Total de usuários: {Usuario.total()}")
    print(f"📊 Total de jogos criados: {Jogo.total()}")
    print(f"📊 Total de plataformas: {Plataforma.total()}")

❌ Nenhum jogo removido.

═══ TODOS OS JOGOS ═══
● Hollow Knight [❤️] | Metroidvania | Single-player | Steam | 40h | ✅ Zerado
   ⭐ Nota: 9.0/10 — Obra de arte, vale cada centavo. (👍 Recomendado)
   📊 Progresso da campanha: 100.0%

● God of War | Ação/Aventura | Single-player | PS5 | 25h | ✅ Zerado
   ⭐ Nota: 10.0/10 — Narrativa impecável. (👍 Recomendado)
   📊 Progresso da campanha: 100.0%

● Hades [❤️] | Roguelike | Single-player | Steam | 60h | ✅ Zerado
   ⭐ Nota: 10.0/10 — Impossível parar de jogar. (👍 Recomendado)
   🔁 Runs totais: 47

● Resident Evil 4 | Survival Horror | Single-player | Xbox | 15h | ⚠️ Em andamento
   📊 Progresso da campanha: 93.8%

● Valorant | FPS Tático | Multiplayer | Steam | 121h | ⚠️ Em andamento
   🎯 Partidas jogadas: 301


═══ EM ANDAMENTO ═══
● Resident Evil 4 | Survival Horror | Single-player | Xbox | 15h | ⚠️ Em andamento
   📊 Progresso da campanha: 93.8%

● Valorant | FPS Tático | Multiplayer | Steam | 121h | ⚠️ Em andamento
   🎯 Partidas jogadas: 301

